# IE 7615 – Deep Learning for AI
## Milestone 2: Multi-Object Detection with YOLOv8

**Goal:** given an image containing several objects, report the **Object ID** and **location** (bounding box) of every object.

**Pipeline**
1. **Part A – Single-object data:** the final merged dataset (73 Object IDs) is cleaned exactly as in Milestone 1 (validation, exact-duplicate removal, resize to 224×224, stratified 70/15/15 split). Each image also gets a tight bounding box around its object (OWLv2, as in Milestone 1). Milestone 1 outputs on Drive are reused automatically.
2. **Part B – Multi-object dataset:** single-object images are **concatenated** into 640×640 composite images on random grid layouts (1×2 up to 3×3, so 2–9 objects per image), with random scale, position, flips and color jitter. Each object's box is known exactly from where it was placed. Composites for train / val / test are built only from the matching Milestone 1 split, so no single-object photo appears in more than one split.
3. **Part C – Training:** YOLOv8s (COCO-pretrained) is fine-tuned on the composites. Class names are the Object IDs, so the model outputs IDs directly.
4. **Part D – Evaluation:** mAP on the held-out test composites, object-level precision / recall / F1, ID accuracy, exact-image accuracy, and performance vs. number of objects.
5. **Part E – Detection:** `detect_objects(image)` returns every Object ID with its location, plus sample results for the report.

**Run:** Colab → *Runtime → Change runtime type → T4 GPU* → *Run all* (≈ 1 h). If Colab disconnects, *Run all* again: every finished step reloads from Drive, and interrupted training resumes from its last checkpoint.

## Setup

In [ ]:
!pip -q install ultralytics

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # reduces GPU out-of-memory errors
import sys, io, re, json, time, random, shutil, hashlib, warnings, contextlib
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageOps
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, top_k_accuracy_score)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision import transforms, models

Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore", category=UserWarning)
from PIL import ImageEnhance
import yaml, gc

In [ ]:
# Mount Google Drive in Google Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

### Paths and settings

In [ ]:
# Final merged dataset: one folder per Object ID (images_OBJ001, images_OBJ002, ...)
DATASET_ROOT = Path("/content/drive/MyDrive/IE7615 Deep Learning for AI")

# Milestone 1 outputs (preprocessed images, split, bounding boxes) - reused if present, recreated otherwise
M1_ROOT = Path("/content/drive/MyDrive/IE_7615/Milestone_1_Results/Four_Model_Comparison")

# All Milestone 2 outputs (dataset, trained model, results, figures) are saved here
OUTPUT_ROOT = Path("/content/drive/MyDrive/IE_7615/Milestone_2_Results/YOLOv8_MultiObject")

# Fast local working copies (Colab's local disk is much faster than Drive)
WORK_DIR = Path("/content") if IN_COLAB else Path.home() / "milestone1_work"   # same local copy as Milestone 1
DATA_DIR = WORK_DIR / "prep_224"        # preprocessed 224x224 images
MO_DIR   = WORK_DIR / "multiobject_ds"  # generated multi-object dataset (YOLO format)

PREP_OUT, BOX_DIR = M1_ROOT / "preprocessing", M1_ROOT / "detection"
TRAIN_DIR, EVAL_DIR, FIG_DIR = OUTPUT_ROOT / "training", OUTPUT_ROOT / "evaluation", OUTPUT_ROOT / "figures"
for d in (OUTPUT_ROOT, PREP_OUT, BOX_DIR, TRAIN_DIR, EVAL_DIR, FIG_DIR, WORK_DIR): d.mkdir(parents=True, exist_ok=True)

SEED        = 42
IMG_SIZE    = 224
VAL_FRAC    = 0.15
TEST_FRAC   = 0.15
RESIZE_MODE = "pad"    # non-square images: letterbox (keeps the whole object)
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_NUMBER_OF_CLASSES = 73

def find_dataset_root(start, max_depth=3):
    # Find the folder containing the images_OBJ### subfolders (handles shortcuts with a different name)
    if not start.exists(): return None
    for root, dirs, _ in os.walk(start):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        if sum(bool(re.search(r"OBJ\d+$", d, re.I)) for d in dirs) >= 10: return Path(root)
        if len(Path(root).relative_to(start).parts) >= max_depth: dirs[:] = []
    return None

if not DATASET_ROOT.exists():
    print("Dataset not at", DATASET_ROOT, "- searching Drive for the images_OBJ### folders ...")
    for start in [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]:
        found = find_dataset_root(start)
        if found: DATASET_ROOT = found; break

print("Dataset path:", DATASET_ROOT, "| exists:", DATASET_ROOT.exists())
print("Output path: ", OUTPUT_ROOT)
if not DATASET_ROOT.exists():
    mydrive = Path("/content/drive/MyDrive")
    print("\nFolders in your My Drive:", sorted(p.name for p in mydrive.iterdir() if p.is_dir()) if mydrive.exists() else "Drive not mounted")
assert DATASET_ROOT.exists(), ("Dataset not found. Open the dataset link, click the folder name > Organize > Add shortcut > My Drive, "
                               "then rerun. Or set DATASET_ROOT to the folder that contains images_OBJ001, images_OBJ002, ...")

### Final Object IDs (verified against the shared sheet)

In [ ]:
OBJECT_NAMES = {
    'OBJ001': 'Basketball', 'OBJ002': 'water bottle', 'OBJ003': 'NoteBook', 'OBJ004': 'Toy Car',
    'OBJ005': 'Computer mouse', 'OBJ006': 'Capybara Toy', 'OBJ007': 'Apple', 'OBJ008': 'Cat',
    'OBJ009': 'Marshall Speaker', 'OBJ010': 'Nike sneakers', 'OBJ011': 'mug', 'OBJ012': 'Computer Keyboard',
    'OBJ013': 'Mobile phone', 'OBJ014': 'Sunglasses', 'OBJ015': 'Water Tumbler', 'OBJ016': 'Sandisk Portable SSD',
    'OBJ017': 'Controller', 'OBJ018': 'Rubiks cube', 'OBJ019': 'Plant', 'OBJ020': 'Airpod',
    'OBJ021': 'Toy Bunny', 'OBJ022': 'Guitar', 'OBJ023': 'Bracelet', 'OBJ024': 'Plush doll',
    'OBJ025': 'Watch', 'OBJ026': 'Red Sox Cap', 'OBJ027': 'Tennis ball', 'OBJ028': 'banana',
    'OBJ029': 'Orange', 'OBJ030': 'Tennis racket', 'OBJ031': 'UNO card box', 'OBJ032': 'Sound Card',
    'OBJ033': 'Duck', 'OBJ034': 'Table Tennis Racket', 'OBJ035': 'Badminton racket', 'OBJ036': 'Pencil case',
    'OBJ037': 'Candle', 'OBJ038': 'Perfume bottle', 'OBJ039': 'Scissors', 'OBJ040': 'Clutcher',
    'OBJ041': 'Table fan', 'OBJ042': 'Mouth guard case', 'OBJ043': 'Swimming Goggels', 'OBJ044': 'Anime Figurine',
    'OBJ045': 'Wayfarer Sunglasses', 'OBJ046': 'iWatch', 'OBJ047': 'Backpack', 'OBJ048': 'Wireless powerbank',
    'OBJ049': 'Pickleball bat', 'OBJ050': 'Hairbrush', 'OBJ051': 'tissue', 'OBJ052': 'Spoon',
    'OBJ053': 'Moisturizer', 'OBJ054': 'Purse', 'OBJ055': 'Scientific Calculator', 'OBJ056': "Trader Joe's Bag",
    'OBJ057': 'Power Bank', 'OBJ058': 'Footstool', 'OBJ059': 'Comb', 'OBJ060': 'Sleep Mask',
    'OBJ061': 'Card wallet', 'OBJ062': 'Headphones', 'OBJ063': 'Measuring Bottle', 'OBJ064': 'RGB Projector',
    'OBJ065': 'Analog watch', 'OBJ066': 'hair oil', 'OBJ067': "Men's Wallet", 'OBJ068': 'Neck Pillow',
    'OBJ069': 'knife', 'OBJ070': 'Astronaut projector', 'OBJ071': 'Sloth Plushie', 'OBJ072': 'Dog',
    'OBJ073': 'Bowl',
}

def disp(obj_id, maxlen=22):
    name = OBJECT_NAMES.get(obj_id)
    return f"{obj_id} · {name[:maxlen]}" if name else obj_id
print(len(OBJECT_NAMES), "Object IDs in the official list")

In [ ]:
# Device, reproducibility and shared helpers
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

if torch.cuda.is_available():            DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():  DEVICE = torch.device("mps")
else:                                    DEVICE = torch.device("cpu")
def amp():          # mixed precision on NVIDIA GPUs
    return torch.autocast("cuda") if DEVICE.type == "cuda" else contextlib.nullcontext()
def sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    elif DEVICE.type == "mps": torch.mps.synchronize()
def free_mem():
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    elif DEVICE.type == "mps": torch.mps.empty_cache()
NUM_WORKERS = 2 if (IN_COLAB or sys.platform.startswith("linux")) else 0
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
if DEVICE.type == "cpu": print("WARNING: no GPU - set Runtime > Change runtime type > T4 GPU, otherwise training will take many hours.")

---
# Part A — Single-object images (same preprocessing as Milestone 1)
These cells reuse the cached Milestone 1 results on Drive (inventory, preprocessed images, split, boxes), so on a normal run they finish in a few minutes.

In [ ]:
OBJ_RE = re.compile(r"(?:^|[^A-Za-z])OBJ\s*0*(\d+)$", re.IGNORECASE)   # "images_OBJ001" -> OBJ001

def infer_label(p: Path):
    for part in p.relative_to(DATASET_ROOT).parts[:-1]:
        m = OBJ_RE.search(part)
        if m: return f"OBJ{int(m.group(1)):03d}"
    return None

all_files = [p for p in DATASET_ROOT.rglob("*") if p.is_file()
             and not any(part.startswith(".") for part in p.relative_to(DATASET_ROOT).parts)]
files = [p for p in all_files if infer_label(p)]
ignored = [p for p in all_files if not infer_label(p)]
if ignored: print(f"Ignored {len(ignored)} files outside Object ID folders, e.g.", [p.name for p in ignored[:5]])

inv = pd.DataFrame({"path": [str(p) for p in files], "label": [infer_label(p) for p in files],
                    "ext": [p.suffix.lower() for p in files]}).sort_values(["label", "path"]).reset_index(drop=True)
non_img = inv[~inv.ext.isin(IMAGE_EXTS)]
folder_ids = sorted(inv.label.unique())

print(f"Files found: {len(inv)} | Object ID folders: {len(folder_ids)}")
print("Extensions:", dict(Counter(inv.ext)))
if len(non_img): print(f"Non-image files (excluded): {len(non_img)}")
print("In dataset but NOT in official list:", sorted(set(folder_ids) - set(OBJECT_NAMES)) or "none")
print("In official list but NOT in dataset:", sorted(set(OBJECT_NAMES) - set(folder_ids)) or "none")
assert len(folder_ids) == EXPECTED_NUMBER_OF_CLASSES, f"Expected {EXPECTED_NUMBER_OF_CLASSES} Object IDs, found {len(folder_ids)}"
print("Object IDs verified.")

In [ ]:
INV_CACHE = PREP_OUT / "raw_inventory.csv"

def inspect_file(path):
    rec = {"path": path, "ok": False, "md5": None, "width": None, "height": None, "mode": None, "error": None}
    try:
        data = Path(path).read_bytes()
        rec["md5"] = hashlib.md5(data).hexdigest()
        with Image.open(io.BytesIO(data)) as im: im.verify()
        with Image.open(io.BytesIO(data)) as im:
            w, h = im.size
            if im.getexif().get(0x0112, 1) in (5, 6, 7, 8): w, h = h, w
            rec.update(width=w, height=h, mode=im.mode, ok=True)
    except Exception as e:
        rec["error"] = repr(e)[:200]
    return rec

img_inv = inv[inv.ext.isin(IMAGE_EXTS)]
if INV_CACHE.exists() and set(pd.read_csv(INV_CACHE).path) == set(img_inv.path):
    meta_df = pd.read_csv(INV_CACHE); print("Loaded cached inventory.")
else:
    with ThreadPoolExecutor(16) as ex:
        meta_df = pd.DataFrame(list(tqdm(ex.map(inspect_file, img_inv.path), total=len(img_inv), desc="Validating")))
    meta_df = img_inv.merge(meta_df, on="path"); meta_df.to_csv(INV_CACHE, index=False)

okm = meta_df.ok.astype(bool); good = meta_df[okm]
print(f"Readable: {okm.sum()} | Unreadable/corrupt: {(~okm).sum()}")
if (~okm).any(): display(meta_df[~okm][["label", "path", "error"]])
print("Most common dimensions:", Counter(zip(good.width.astype(int), good.height.astype(int))).most_common(5))
not224 = good[(good.width != IMG_SIZE) | (good.height != IMG_SIZE)]
print(f"Images not {IMG_SIZE}x{IMG_SIZE}: {len(not224)} (resized, not dropped)")
print("Color modes:", dict(Counter(good["mode"])))

In [ ]:
ok = good.copy()
ok["dup_rank"] = ok.groupby(["label", "md5"]).cumcount()
lab_per_hash = ok.groupby("md5")["label"].nunique()
cross = set(lab_per_hash[lab_per_hash > 1].index)       # same image under 2+ Object IDs -> ambiguous label

excl  = [(r.path, r.label, "not an image file") for r in non_img.itertuples()]
excl += [(r.path, r.label, "unreadable/corrupt") for r in meta_df[~okm].itertuples()]
excl += [(r.path, r.label, "exact duplicate within class") for r in ok[ok.dup_rank > 0].itertuples()]
excl += [(r.path, r.label, "exact duplicate across classes (ambiguous label)") for r in ok[(ok.dup_rank == 0) & ok.md5.isin(cross)].itertuples()]
excluded_df = pd.DataFrame(excl, columns=["path", "label", "reason"])
excluded_df.to_csv(PREP_OUT / "excluded_images.csv", index=False)

clean = ok[(ok.dup_rank == 0) & ~ok.md5.isin(cross)].reset_index(drop=True)
print(f"Clean images: {len(clean)} / {len(inv)} files")
print(excluded_df.reason.value_counts().to_string() if len(excluded_df) else "nothing excluded")
counts = clean.label.value_counts().sort_index()
print(f"Images per Object ID: min={counts.min()}  median={int(counts.median())}  max={counts.max()}")

fig, ax = plt.subplots(figsize=(18, 4))
ax.bar(counts.index, counts.values); ax.set_title("Clean images per Object ID"); ax.set_ylabel("images")
ax.tick_params(axis="x", rotation=90, labelsize=7); plt.tight_layout()
plt.savefig(PREP_OUT / "class_distribution.png", dpi=150); plt.show()

In [ ]:
PREP_ZIP = PREP_OUT / "prep_224.zip"
clean["rel_path"] = clean.label + "/" + clean.md5 + ".jpg"
if not DATA_DIR.exists() and PREP_ZIP.exists():
    print("Restoring preprocessed images from Drive..."); shutil.unpack_archive(str(PREP_ZIP), str(DATA_DIR.parent))

def load_and_resize(path, size=IMG_SIZE, mode=RESIZE_MODE):
    im = Image.open(path)
    if im.format == "JPEG": im.draft("RGB", (size * 2, size * 2))      # fast decode for very large JPEGs
    im = ImageOps.exif_transpose(im)
    if im.mode in ("RGBA", "LA", "P"):
        im = im.convert("RGBA"); bg = Image.new("RGBA", im.size, (255, 255, 255, 255)); bg.alpha_composite(im); im = bg
    im = im.convert("RGB")
    w, h = im.size
    if abs(w / h - 1) < 0.02: return im.resize((size, size), Image.Resampling.LANCZOS)
    if mode == "crop": return ImageOps.fit(im, (size, size), Image.Resampling.LANCZOS)
    return ImageOps.pad(im, (size, size), Image.Resampling.LANCZOS, color=(0, 0, 0))

def prep_one(row):
    out = DATA_DIR / row["rel_path"]
    if not out.exists():
        out.parent.mkdir(parents=True, exist_ok=True)
        load_and_resize(row["path"]).save(out, quality=95)
    return True

missing = [r for r in clean.to_dict("records") if not (DATA_DIR / r["rel_path"]).exists()]
if missing:
    with ThreadPoolExecutor(16) as ex:
        list(tqdm(ex.map(prep_one, missing), total=len(missing), desc="Resizing"))
    shutil.make_archive(str(PREP_ZIP.with_suffix("")), "zip", root_dir=str(DATA_DIR.parent), base_dir=DATA_DIR.name)
print("Preprocessed images ready:", len(clean))
clean["img"] = [str(DATA_DIR / p) for p in clean.rel_path]

rng = np.random.default_rng(SEED)
show = rng.choice(folder_ids, size=min(24, len(folder_ids)), replace=False)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, lab in zip(axes.flat, show):
    ax.imshow(Image.open(clean[clean.label == lab].sample(1, random_state=SEED).img.iloc[0]))
    ax.set_title(disp(lab, 16), fontsize=8); ax.axis("off")
for ax in axes.flat[len(show):]: ax.axis("off")
plt.tight_layout(); plt.savefig(PREP_OUT / "sample_images.png", dpi=120); plt.show()

In [ ]:
classes = sorted(clean.label.unique()); NUM_CLASSES = len(classes)
clean["y"] = clean.label.map({c: i for i, c in enumerate(classes)})
train_df, temp_df = train_test_split(clean, test_size=VAL_FRAC + TEST_FRAC, stratify=clean.y, random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=temp_df.y, random_state=SEED)
train_df, val_df, test_df = [d.reset_index(drop=True) for d in (train_df, val_df, test_df)]
assert not (set(train_df.md5) & (set(val_df.md5) | set(test_df.md5))), "leakage!"
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert d.label.nunique() == NUM_CLASSES
    d[["rel_path", "label", "y", "md5", "path"]].to_csv(PREP_OUT / f"split_{name}.csv", index=False)
pd.DataFrame({"object_id": classes, "object_name": [OBJECT_NAMES.get(c, "") for c in classes],
              "clean_images": counts.reindex(classes).values}).to_csv(PREP_OUT / "object_ids.csv", index=False)

prep_summary = {"raw_files": int(len(inv)), "readable_images": int(okm.sum()), "clean_images": int(len(clean)),
                "excluded_by_reason": excluded_df.reason.value_counts().to_dict(), "images_not_224": int(len(not224)),
                "images_per_class": {"min": int(counts.min()), "median": int(counts.median()), "max": int(counts.max())},
                "num_classes": NUM_CLASSES, "img_size": IMG_SIZE, "resize_mode_non_square": RESIZE_MODE,
                "split_sizes": {"train": len(train_df), "val": len(val_df), "test": len(test_df)}, "seed": SEED}
json.dump(prep_summary, open(PREP_OUT / "preprocessing_summary.json", "w"), indent=2)
prep_summary

### A6. Tight bounding box for each single-object image
OWLv2 locates the object inside each 224×224 photo (prompt: *"a photo of a <object name>"*). These boxes were already generated in Milestone 1 and are loaded from cache. If OWLv2 was not confident, the whole photo is used as the box (fallback).

In [ ]:
OWL_ID, BOX_CACHE = "google/owlv2-base-patch16-ensemble", BOX_DIR / "pseudo_boxes.csv"
MIN_SCORE, MIN_AREA, OWL_BATCH = 0.10, 0.005, 4

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
done = pd.read_csv(BOX_CACHE) if BOX_CACHE.exists() else pd.DataFrame(columns=["rel_path"])
todo = all_df[~all_df.rel_path.isin(set(done.rel_path))]
print(f"Boxes cached: {len(done)} | to generate: {len(todo)}" + ("  -> all loaded from Milestone 1, nothing to compute" if len(todo) == 0 else ""))

def prompt_for(label):
    name = OBJECT_NAMES.get(label)
    return [f"a photo of a {name}" if name else "an object", "an object"]

if len(todo):
    from transformers import Owlv2Processor, Owlv2ForObjectDetection
    proc = Owlv2Processor.from_pretrained(OWL_ID)
    owl = Owlv2ForObjectDetection.from_pretrained(OWL_ID).to(DEVICE).eval()
    S, rows = IMG_SIZE, []
    for bi, i in enumerate(tqdm(range(0, len(todo), OWL_BATCH), desc="OWLv2 boxes")):
        chunk = todo.iloc[i:i + OWL_BATCH]
        imgs = [Image.open(p).convert("RGB") for p in chunk.img]
        inputs = proc(text=[prompt_for(l) for l in chunk.label], images=imgs, return_tensors="pt").to(DEVICE)
        with torch.no_grad(), amp(): out = owl(**inputs)
        scores = out.logits.float().sigmoid().max(-1).values
        best = scores.argmax(1)
        for b, r in enumerate(chunk.itertuples()):
            s = scores[b, best[b]].item()
            cx, cy, w, h = out.pred_boxes[b, best[b]].float().tolist()       # normalized; images are square
            x1, y1, x2, y2 = max(0., (cx - w/2) * S), max(0., (cy - h/2) * S), min(S, (cx + w/2) * S), min(S, (cy + h/2) * S)
            fallback = s < MIN_SCORE or (x2 - x1) * (y2 - y1) / S**2 < MIN_AREA or x2 - x1 < 2 or y2 - y1 < 2
            if fallback: x1, y1, x2, y2 = 0., 0., float(S), float(S)
            rows.append({"rel_path": r.rel_path, "x1": x1, "y1": y1, "x2": x2, "y2": y2, "owl_score": s, "fallback": fallback})
        if bi % 50 == 0 or i + OWL_BATCH >= len(todo):
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); rows = []; done.to_csv(BOX_CACHE, index=False)
    del owl; free_mem()

boxes = pd.read_csv(BOX_CACHE).drop_duplicates("rel_path")
det_train, det_val, det_test = [d.merge(boxes, on="rel_path", how="left") for d in (train_df, val_df, test_df)]
assert not pd.concat([det_train, det_val, det_test]).x1.isna().any(), "some images have no box"
allb = pd.concat([det_train, det_val, det_test]); allb["fallback"] = allb.fallback.astype(str).str.lower() == "true"
for d in (det_train, det_val, det_test): d["fallback"] = d.fallback.astype(str).str.lower() == "true"
fb_rate = allb.fallback.mean()
print(f"Full-image fallback boxes: {fb_rate:.1%} of images")
fb = allb.groupby("label").fallback.mean().sort_values(ascending=False)
print("Object IDs with the most fallbacks:"); print(fb.head(10).rename(index=disp).map("{:.0%}".format).to_string())
fb.to_csv(BOX_DIR / "pseudo_box_fallback_by_class.csv")

s = det_train.sample(24, random_state=SEED)
fig, axes = plt.subplots(3, 8, figsize=(18, 7))
for ax, r in zip(axes.flat, s.itertuples()):
    ax.imshow(Image.open(r.img)); ax.axis("off")
    ax.add_patch(patches.Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=2, edgecolor="orange" if r.fallback else "lime"))
    ax.set_title(f"{disp(r.label, 14)}\nscore {r.owl_score:.2f}", fontsize=7)
fig.suptitle("Generated boxes (green = OWLv2, orange = full-image fallback)"); plt.tight_layout()
plt.savefig(BOX_DIR / "pseudo_boxes_examples.png", dpi=120); plt.show()

---
# Part B — Multi-object dataset by concatenating single-object images

**How each composite is made**
1. Pick a grid layout at random: 1×2, 2×1, 1×3, 3×1, 2×2, 2×3, 3×2 or 3×3 (larger grids weighted higher).
2. Leave each cell empty with 15% probability (at least 2 objects per image), so object counts and positions vary.
3. Fill each cell with a single-object image drawn from a **class-balanced pool** of that split (every photo is used before any repeats; no Object ID appears twice in the same composite).
4. **Augment** each object: random scale (55–100% of the cell), random position within the cell, horizontal flip (50%), brightness / contrast / saturation jitter (±20%). Background color is random.
5. The object's box is the single-image box transformed to its pasted position, so labels are exact by construction.

YOLOv8's own training augmentations (mosaic, HSV, translation, scaling, flips) are applied on top during training.

In [ ]:
CANVAS       = 640
N_IMAGES     = {"train": 3000, "val": 500, "test": 500}       # requirement: at least 100
LAYOUTS      = {(1, 2): 1, (2, 1): 1, (1, 3): 1, (3, 1): 1, (2, 2): 3, (2, 3): 3, (3, 2): 3, (3, 3): 3}   # (rows, cols): weight
EMPTY_CELL_P = 0.15
SCALE_RANGE  = (0.55, 1.0)
BOX_MODE     = "tight"      # "tight" = OWLv2 box inside each photo; "tile" = whole pasted photo

class BalancedPool:
    # Cycles through every image of a split in random order; skips Object IDs already in the current composite.
    def __init__(self, df, rng):
        self.rows, self.rng, self.queue = df.to_dict("records"), rng, []
    def draw(self, exclude):
        for _ in range(2):
            for k in range(len(self.queue) - 1, -1, -1):      # next unused photo whose Object ID isn't in this composite yet
                if self.queue[k]["label"] not in exclude: return self.queue.pop(k)
            fresh = self.rows[:]; self.rng.shuffle(fresh)     # cycle finished (or only excluded IDs left): start a new one
            self.queue = fresh + self.queue
        raise RuntimeError("could not draw a new Object ID")

def color_jitter(img, rng, strength=0.2):
    for enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
        img = enh(img).enhance(rng.uniform(1 - strength, 1 + strength))
    return img

def make_composite(pool, rng, layout=None, empty_p=EMPTY_CELL_P):
    rows, cols = layout or rng.choices(list(LAYOUTS), weights=list(LAYOUTS.values()))[0]
    cw, ch = CANVAS / cols, CANVAS / rows
    canvas = Image.new("RGB", (CANVAS, CANVAS), tuple(rng.randint(90, 245) for _ in range(3)))
    cells = [(r, c) for r in range(rows) for c in range(cols)]
    filled = [cell for cell in cells if rng.random() >= empty_p]
    while len(filled) < min(2, len(cells)):
        extra = rng.choice([cell for cell in cells if cell not in filled]); filled.append(extra)
    objs, used = [], set()
    for (r, c) in sorted(filled):
        it = pool.draw(used); used.add(it["label"])
        side = max(32, int(min(cw, ch) * rng.uniform(*SCALE_RANGE)))
        img = Image.open(it["img"]).convert("RGB")
        flip = rng.random() < 0.5
        if flip: img = ImageOps.mirror(img)
        img = color_jitter(img, rng).resize((side, side), Image.Resampling.BILINEAR)
        ox = int(c * cw + rng.uniform(0, cw - side)); oy = int(r * ch + rng.uniform(0, ch - side))
        canvas.paste(img, (ox, oy))
        bx1, by1, bx2, by2 = (it["x1"], it["y1"], it["x2"], it["y2"]) if BOX_MODE == "tight" else (0, 0, IMG_SIZE, IMG_SIZE)
        if flip: bx1, bx2 = IMG_SIZE - bx2, IMG_SIZE - bx1
        k = side / IMG_SIZE
        objs.append({"label": it["label"], "y": int(it["y"]), "src": it["rel_path"], "layout": f"{rows}x{cols}",
                     "x1": ox + bx1 * k, "y1": oy + by1 * k, "x2": ox + bx2 * k, "y2": oy + by2 * k})
    return canvas, objs

def yolo_line(o):
    cx, cy = (o["x1"] + o["x2"]) / 2 / CANVAS, (o["y1"] + o["y2"]) / 2 / CANVAS
    w, h = (o["x2"] - o["x1"]) / CANVAS, (o["y2"] - o["y1"]) / CANVAS
    return f"{o['y']} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"

In [ ]:
MO_ZIP, MANIFEST = OUTPUT_ROOT / "multiobject_dataset.zip", OUTPUT_ROOT / "multiobject_manifest.csv"
sources = {"train": det_train, "val": det_val, "test": det_test}

if not (MO_DIR / "test" / "labels").exists() and MO_ZIP.exists() and MANIFEST.exists():
    print("Restoring generated dataset from Drive ..."); shutil.unpack_archive(str(MO_ZIP), str(MO_DIR.parent))

if not (MO_DIR / "test" / "labels").exists() or not MANIFEST.exists():
    rng_gen, records = random.Random(SEED), []
    for split, n in N_IMAGES.items():
        (MO_DIR / split / "images").mkdir(parents=True, exist_ok=True); (MO_DIR / split / "labels").mkdir(parents=True, exist_ok=True)
        pool = BalancedPool(sources[split], rng_gen)
        for i in tqdm(range(n), desc=f"Generating {split}"):
            canvas, objs = make_composite(pool, rng_gen)
            name = f"mo_{split}_{i:05d}"
            canvas.save(MO_DIR / split / "images" / f"{name}.jpg", quality=92)
            (MO_DIR / split / "labels" / f"{name}.txt").write_text("\n".join(yolo_line(o) for o in objs) + "\n")
            records += [{"split": split, "image": name, **o} for o in objs]
    pd.DataFrame(records).to_csv(MANIFEST, index=False)
    shutil.make_archive(str(MO_ZIP.with_suffix("")), "zip", root_dir=str(MO_DIR.parent), base_dir=MO_DIR.name)
    print("Saved dataset to", MO_ZIP)

DATA_YAML = MO_DIR / "data.yaml"
yaml.safe_dump({"path": str(MO_DIR.resolve()), "train": "train/images", "val": "val/images", "test": "test/images",
                "names": {i: c for i, c in enumerate(classes)}}, open(DATA_YAML, "w"), sort_keys=False)
manifest = pd.read_csv(MANIFEST)

# leakage check: no single-object photo is used in two splits
src_split = manifest.groupby("src").split.nunique()
assert (src_split == 1).all(), "a source photo appears in more than one split!"

per_img = manifest.groupby(["split", "image"]).size()
inst = manifest[manifest.split == "train"].label.value_counts()
mo_summary = {s: {"images": int(manifest[manifest.split == s].image.nunique()), "objects": int((manifest.split == s).sum()),
                  "objects_per_image_mean": round(float(per_img[s].mean()), 2), "objects_per_image_range": [int(per_img[s].min()), int(per_img[s].max())]}
              for s in N_IMAGES}
mo_summary["train_instances_per_object_id"] = {"min": int(inst.min()), "median": int(inst.median()), "max": int(inst.max())}
mo_summary["object_ids_in_each_split"] = {s: int(manifest[manifest.split == s].label.nunique()) for s in N_IMAGES}
mo_summary["box_mode"] = BOX_MODE
json.dump(mo_summary, open(OUTPUT_ROOT / "multiobject_dataset_summary.json", "w"), indent=2)
mo_summary

In [ ]:
PALETTE = [plt.cm.tab20(i % 20) for i in range(NUM_CLASSES)]
def draw(ax, img, boxes, ids, scores=None, colors=None, fs=7):
    ax.imshow(img); ax.axis("off")
    for k, ((x1, y1, x2, y2), oid) in enumerate(zip(boxes, ids)):
        col = colors[k] if colors is not None else PALETTE[classes.index(oid) if oid in classes else 0]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2, edgecolor=col))
        txt = oid if scores is None else f"{oid} {scores[k]:.2f}"
        ax.text(x1 + 2, y1 + 2, txt, fontsize=fs, color="white", va="top", bbox=dict(facecolor=col, alpha=.85, pad=1, lw=0))

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, name in zip(axes.flat, manifest[manifest.split == "train"].image.drop_duplicates().sample(8, random_state=SEED)):
    g = manifest[manifest.image == name]
    draw(ax, Image.open(MO_DIR / "train" / "images" / f"{name}.jpg"), g[["x1", "y1", "x2", "y2"]].values, g.label.tolist())
    ax.set_title(f"{name} · {g.layout.iloc[0]} grid · {len(g)} objects", fontsize=9)
fig.suptitle("Generated multi-object training images with ground-truth Object IDs and boxes"); plt.tight_layout()
plt.savefig(FIG_DIR / "generated_samples.png", dpi=110); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
per_img["train"].value_counts().sort_index().plot.bar(ax=axes[0], color="#2E5C8A"); axes[0].set_title("Objects per training image"); axes[0].set_xlabel("objects")
inst.sort_index().plot.bar(ax=axes[1], color="#8FB3D9"); axes[1].set_title("Training instances per Object ID"); axes[1].tick_params(axis="x", labelsize=5)
plt.tight_layout(); plt.savefig(FIG_DIR / "dataset_statistics.png", dpi=130); plt.show()

---
# Part C — Train YOLOv8s
COCO-pretrained YOLOv8s, fine-tuned at 640×640. Early stopping (patience 20) keeps the best checkpoint by validation mAP. Rotation is disabled because the boxes are axis-aligned; mosaic is reduced to 50% since the images are already composites, and turned off for the final 10 epochs.

In [ ]:
from ultralytics import YOLO
MODEL_NAME, EPOCHS, BATCH = "yolov8s", 80, 16      # 16 fits a 15 GB T4 comfortably at 640 px
RUN = f"{MODEL_NAME}_multiobject"
RUN_DIR = TRAIN_DIR / RUN
BEST, LAST, DONE = RUN_DIR / "weights" / "best.pt", RUN_DIR / "weights" / "last.pt", TRAIN_DIR / f"{RUN}_done.json"
YOLO_DEVICE = 0 if DEVICE.type == "cuda" else DEVICE.type

def free_gpu():
    gc.collect(); free_mem()

def run_training(batch):
    if LAST.exists():
        print(f"Resuming interrupted training from the last checkpoint (batch {batch}) ...")
        YOLO(str(LAST)).train(resume=True, batch=batch)
    else:
        YOLO(f"{MODEL_NAME}.pt").train(
            data=str(DATA_YAML), imgsz=CANVAS, epochs=EPOCHS, batch=batch, patience=20, seed=SEED, deterministic=True,
            workers=NUM_WORKERS, device=YOLO_DEVICE, project=str(TRAIN_DIR), name=RUN, exist_ok=True,
            degrees=0.0, mosaic=0.5, close_mosaic=10, plots=True, verbose=False)

if DONE.exists() and BEST.exists():
    print("Loaded the trained model from Drive.")
else:
    for b in (BATCH, BATCH // 2, BATCH // 4):
        try:
            free_gpu(); run_training(b); BATCH = b; break
        except torch.cuda.OutOfMemoryError:
            print(f"Out of GPU memory with batch {b} - retrying with batch {b // 2} ..."); free_gpu()
    else:
        raise RuntimeError("Out of GPU memory even with a small batch: use Runtime > Restart session, then Run all.")
    json.dump({"finished": True, "batch": BATCH}, open(DONE, "w"))
free_gpu()

model = YOLO(str(BEST))
hist = pd.read_csv(RUN_DIR / "results.csv"); hist.columns = [c.strip() for c in hist.columns]
train_minutes = float(hist["time"].iloc[-1]) / 60 if "time" in hist.columns else float("nan")
best_epoch = int(hist.loc[hist["metrics/mAP50-95(B)"].idxmax(), "epoch"])
print(f"Epochs run: {len(hist)} | best epoch: {best_epoch} | training time: {train_minutes:.1f} min")
shutil.copy(BEST, OUTPUT_ROOT / f"{RUN}_best.pt")      # final weights (for predict.py / GitHub release)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.2))
for col, lab in [("train/box_loss", "train box"), ("val/box_loss", "val box"), ("train/cls_loss", "train cls"), ("val/cls_loss", "val cls")]:
    axes[0].plot(hist.epoch, hist[col], "--" if col.startswith("train") else "-", label=lab)
axes[0].set_title("Losses"); axes[0].legend(fontsize=8)
axes[1].plot(hist.epoch, hist["metrics/mAP50(B)"], label="mAP@0.5"); axes[1].plot(hist.epoch, hist["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
axes[1].set_title("Validation mAP"); axes[1].legend(fontsize=8)
axes[2].plot(hist.epoch, hist["metrics/precision(B)"], label="precision"); axes[2].plot(hist.epoch, hist["metrics/recall(B)"], label="recall")
axes[2].set_title("Validation precision / recall"); axes[2].legend(fontsize=8)
for ax in axes: ax.set_xlabel("epoch"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=150); plt.show()

---
# Part D — Evaluation on the held-out test set (500 composites built only from test-split photos)

- **mAP@0.5 / mAP@0.5:0.95** — standard COCO-style detection metrics (Ultralytics).
- **Object-level precision / recall / F1** at the deployment confidence (0.25): a detection is correct only if it has the **right Object ID** and overlaps the true box with IoU ≥ 0.5.
- **Localization recall** (any ID, IoU ≥ 0.5) and **ID accuracy** among localized objects separate "found it" from "named it correctly".
- **Exact-image accuracy:** share of test images where *every* object was found with the correct ID and nothing extra was predicted.

In [ ]:
CONF, NMS_IOU, MATCH_IOU = 0.25, 0.6, 0.5

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    a, b = np.asarray(a, float)[:, None, :], np.asarray(b, float)[None, :, :]
    iw = np.clip(np.minimum(a[..., 2], b[..., 2]) - np.maximum(a[..., 0], b[..., 0]), 0, None)
    ih = np.clip(np.minimum(a[..., 3], b[..., 3]) - np.maximum(a[..., 1], b[..., 1]), 0, None)
    inter = iw * ih
    area = lambda x: (x[..., 2] - x[..., 0]) * (x[..., 3] - x[..., 1])
    return inter / (area(a) + area(b) - inter + 1e-9)

def greedy_match(gt_boxes, gt_cls, pb, pc, ps, same_class):
    # Greedy matching by prediction confidence. Returns list of (gt_idx, pred_idx) pairs.
    ious, pairs, used_g = iou_matrix(gt_boxes, pb), [], set()
    for j in np.argsort(-np.asarray(ps)):
        best, best_i = MATCH_IOU, None
        for i in range(len(gt_boxes)):
            if i in used_g or (same_class and gt_cls[i] != pc[j]): continue
            if ious[i, j] >= best: best, best_i = ious[i, j], i
        if best_i is not None: pairs.append((best_i, j)); used_g.add(best_i)
    return pairs

def evaluate_objects(gt_df, preds):
    tp = Counter(); fp = Counter(); fn = Counter(); confusions = Counter()
    rows, n_loc, n_loc_correct = [], 0, 0
    for name, g in gt_df.groupby("image"):
        gb, gc = g[["x1", "y1", "x2", "y2"]].values, g.y.values.astype(int)
        pb, pc, ps = preds.get(name, (np.zeros((0, 4)), np.zeros(0, int), np.zeros(0)))
        m = greedy_match(gb, gc, pb, pc, ps, same_class=True)
        mg, mp = {i for i, _ in m}, {j for _, j in m}
        for i, _ in m: tp[gc[i]] += 1
        for i in set(range(len(gb))) - mg: fn[gc[i]] += 1
        for j in set(range(len(pb))) - mp: fp[int(pc[j])] += 1
        loc = greedy_match(gb, gc, pb, pc, ps, same_class=False)
        n_loc += len(loc); n_loc_correct += sum(gc[i] == pc[j] for i, j in loc)
        for i, j in loc:
            if gc[i] != pc[j]: confusions[(classes[gc[i]], classes[int(pc[j])])] += 1
        rows.append({"image": name, "n_gt": len(gb), "n_pred": len(pb), "tp": len(m),
                     "exact": len(m) == len(gb) == len(pb), "count_ok": len(pb) == len(gb), "layout": g.layout.iloc[0]})
    T, P, N = sum(tp.values()), sum(fp.values()), sum(fn.values())
    prec, rec = T / max(1, T + P), T / max(1, T + N)
    img = pd.DataFrame(rows)
    summary = {"objects": int(T + N), "detections": int(T + P), "precision": prec, "recall": rec, "f1": 2 * prec * rec / max(1e-9, prec + rec),
               "localization_recall_any_id": n_loc / max(1, T + N), "id_accuracy_of_localized": float(n_loc_correct / max(1, n_loc)),
               "exact_image_accuracy": float(img.exact.mean()), "object_count_accuracy": float(img.count_ok.mean())}
    per_class = pd.DataFrame({"object_id": classes, "object_name": [OBJECT_NAMES.get(c, "") for c in classes],
                              "tp": [tp[i] for i in range(NUM_CLASSES)], "fp": [fp[i] for i in range(NUM_CLASSES)], "fn": [fn[i] for i in range(NUM_CLASSES)]})
    per_class["precision"] = per_class.tp / (per_class.tp + per_class.fp).clip(lower=1)
    per_class["recall"] = per_class.tp / (per_class.tp + per_class.fn).clip(lower=1)
    per_class["f1"] = 2 * per_class.precision * per_class.recall / (per_class.precision + per_class.recall).replace(0, np.nan)
    return summary, per_class.fillna(0), img, confusions

In [ ]:
# Load the trained model from Drive if this cell is run on its own (e.g. after a session restart)
if "model" not in globals():
    assert BEST.exists(), f"No trained model at {BEST} - run the Part C training cell first."
    model = YOLO(str(BEST)); print("Loaded trained model from", BEST)

# 1) Standard COCO-style mAP on the test split
tm = model.val(data=str(DATA_YAML), split="test", imgsz=CANVAS, batch=BATCH, conf=0.001, iou=NMS_IOU, device=YOLO_DEVICE,
               project=str(EVAL_DIR), name="test", exist_ok=True, plots=True, verbose=False)
coco = {"mAP50": float(tm.box.map50), "mAP50_95": float(tm.box.map), "precision": float(tm.box.mp), "recall": float(tm.box.mr)}

# 2) Object-level evaluation at the deployment confidence threshold
test_imgs = sorted((MO_DIR / "test" / "images").glob("*.jpg"))
gc.collect(); free_mem()
PRED_BATCH = 16      # predict in small chunks: recent Ultralytics versions batch a whole list at once, which runs out of GPU memory
preds, t0 = {}, time.time()
for i in range(0, len(test_imgs), PRED_BATCH):
    chunk = [str(p) for p in test_imgs[i:i + PRED_BATCH]]
    for r in model.predict(source=chunk, imgsz=CANVAS, conf=CONF, iou=NMS_IOU, batch=PRED_BATCH, verbose=False, device=YOLO_DEVICE):
        b = r.boxes
        preds[Path(r.path).stem] = (b.xyxy.cpu().numpy(), b.cls.cpu().numpy().astype(int), b.conf.cpu().numpy())
ms_per_img = (time.time() - t0) / len(test_imgs) * 1000
gt_test = manifest[manifest.split == "test"]
obj, per_class, img_res, confusions = evaluate_objects(gt_test, preds)

ap50 = np.full(NUM_CLASSES, np.nan); ap = np.full(NUM_CLASSES, np.nan)
for k, c in enumerate(tm.box.ap_class_index): ap50[int(c)], ap[int(c)] = tm.box.ap50[k], tm.box.ap[k]
per_class["AP50"], per_class["AP50_95"] = ap50, ap
per_class.to_csv(EVAL_DIR / "per_object_id_results.csv", index=False)

results_table = pd.DataFrame([
    ["mAP@0.5", coco["mAP50"]], ["mAP@0.5:0.95", coco["mAP50_95"]],
    [f"Precision (correct ID & IoU≥{MATCH_IOU}, conf≥{CONF})", obj["precision"]], ["Recall", obj["recall"]], ["F1", obj["f1"]],
    ["Localization recall (any ID)", obj["localization_recall_any_id"]], ["ID accuracy of localized objects", obj["id_accuracy_of_localized"]],
    ["Exact-image accuracy (all objects right, nothing extra)", obj["exact_image_accuracy"]], ["Object-count accuracy", obj["object_count_accuracy"]],
    ["Inference time (ms / image, incl. pre/post-processing)", ms_per_img]], columns=["metric", "value"])
results_table.to_csv(EVAL_DIR / "test_results.csv", index=False)
display(results_table.round(4))

In [ ]:
# Performance vs. number of objects per image (more objects -> smaller objects)
by_n = img_res.groupby("n_gt").agg(images=("image", "size"), objects=("n_gt", "sum"), found_correctly=("tp", "sum"), exact=("exact", "mean"))
by_n["recall"] = by_n.found_correctly / by_n.objects
by_n.to_csv(EVAL_DIR / "results_by_object_count.csv"); display(by_n.round(3))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(by_n.index.astype(str), by_n.recall, color="#2E5C8A", label="object recall"); ax.plot(by_n.index.astype(str), by_n.exact, "o-", color="#C0504D", label="exact-image accuracy")
ax.set_ylim(0, 1.05); ax.set_xlabel("objects in image"); ax.legend(); ax.grid(axis="y", alpha=.3); ax.set_title("Test performance vs. number of objects")
plt.tight_layout(); plt.savefig(FIG_DIR / "performance_vs_object_count.png", dpi=150); plt.show()

print("\n8 weakest Object IDs (test F1):")
display(per_class.sort_values("f1").head(8)[["object_id", "object_name", "precision", "recall", "f1", "AP50"]].round(3))
print("\nMost frequent ID confusions (true -> predicted : count):")
for (a, b), n in confusions.most_common(10): print(f"  {disp(a)} -> {disp(b)} : {n}")

---
# Part E — Detect objects in any multi-object image
`detect_objects(image)` returns a table with the **Object ID, object name, confidence and bounding box** (pixel coordinates, top-left origin) of every object found, and draws them on the image.

In [ ]:
def detect_objects(image, conf=CONF, show=True, title=None, ax=None):
    r = model.predict(image, imgsz=CANVAS, conf=conf, iou=NMS_IOU, verbose=False, device=YOLO_DEVICE)[0]
    b = r.boxes
    rows = [{"object_id": r.names[int(c)], "object_name": OBJECT_NAMES.get(r.names[int(c)], ""), "confidence": round(float(s), 3),
             "x1": int(x1), "y1": int(y1), "x2": int(x2), "y2": int(y2), "center_x": int((x1 + x2) / 2), "center_y": int((y1 + y2) / 2)}
            for (x1, y1, x2, y2), c, s in zip(b.xyxy.cpu().numpy(), b.cls.cpu().numpy(), b.conf.cpu().numpy())]
    out = pd.DataFrame(rows, columns=["object_id", "object_name", "confidence", "x1", "y1", "x2", "y2", "center_x", "center_y"])
    out = out.sort_values(["center_y", "center_x"]).reset_index(drop=True)
    if show:
        own = ax is None
        if own: fig, ax = plt.subplots(figsize=(6, 6))
        draw(ax, Image.open(image).convert("RGB") if isinstance(image, (str, Path)) else image,
             out[["x1", "y1", "x2", "y2"]].values, out.object_id.tolist(), out.confidence.tolist())
        ax.set_title(title or f"{len(out)} objects detected", fontsize=9)
        if own: plt.show()
    return out

# Example: one test image
example = str(test_imgs[0])
detect_objects(example, title=f"{Path(example).stem}: predicted Object IDs and locations")

In [ ]:
# Sample detection results for the report: prediction (left) vs. ground truth (right)
sel = img_res.sort_values("image").groupby("n_gt").head(1).image.tolist()[:6]
fig, axes = plt.subplots(len(sel), 2, figsize=(11, 5.4 * len(sel)))
for row, name in zip(axes, sel):
    g = gt_test[gt_test.image == name]
    det = detect_objects(str(MO_DIR / "test" / "images" / f"{name}.jpg"), ax=row[0], title=f"Prediction · {name}")
    draw(row[1], Image.open(MO_DIR / "test" / "images" / f"{name}.jpg"), g[["x1", "y1", "x2", "y2"]].values, g.label.tolist())
    row[1].set_title(f"Ground truth · {len(g)} objects", fontsize=9)
plt.tight_layout(); plt.savefig(FIG_DIR / "sample_detections.png", dpi=100); plt.show()

In [ ]:
# Stress test: a NEW layout never used in training (4x3 grid, 12 objects) built from test-split photos
rng_demo = random.Random(SEED + 1)
canvas, objs = make_composite(BalancedPool(det_test, rng_demo), rng_demo, layout=(4, 3), empty_p=0.0)
canvas.save(FIG_DIR / "stress_test_input.jpg")
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
det = detect_objects(canvas, ax=axes[0], title="Prediction on an unseen 4x3 layout (12 objects)")
draw(axes[1], canvas, [[o["x1"], o["y1"], o["x2"], o["y2"]] for o in objs], [o["label"] for o in objs]); axes[1].set_title("Ground truth")
plt.tight_layout(); plt.savefig(FIG_DIR / "stress_test_12_objects.png", dpi=100); plt.show()
s_obj, *_ = evaluate_objects(pd.DataFrame([{**o, "image": "stress"} for o in objs]),
                             {"stress": (det[["x1", "y1", "x2", "y2"]].values, det.object_id.map(classes.index).values, det.confidence.values)})
print(f"Stress test: {round(s_obj['recall'] * 12)} / 12 objects found with the correct ID")
display(det)

**Try your own image:** upload a photo (Colab sidebar → Files → Upload) and run `detect_objects("/content/your_photo.jpg")`. Note that the model was trained on concatenated product-style photos, so natural scenes with heavy clutter or occlusion are harder.

In [ ]:
# Everything needed for the report. Paste this output into the chat to get the report written.
summary = {"dataset": mo_summary, "single_object_source": {"clean_images": int(len(clean)), "object_ids": NUM_CLASSES,
           "pseudo_box_fallback_rate": round(float(fb_rate), 4)},
           "training": {"model": MODEL_NAME, "img_size": CANVAS, "epochs_run": int(len(hist)), "best_epoch": best_epoch,
                        "train_minutes": round(train_minutes, 1), "batch": BATCH},
           "test_coco": {k: round(v, 4) for k, v in coco.items()},
           "test_object_level": {k: (round(v, 4) if isinstance(v, float) else v) for k, v in obj.items()},
           "ms_per_image": round(ms_per_img, 2),
           "by_object_count": by_n.round(4).reset_index().to_dict(orient="records"),
           "weakest_object_ids": per_class.sort_values("f1").head(8)[["object_id", "object_name", "precision", "recall", "f1"]].round(3).to_dict(orient="records"),
           "top_confusions": [{"true": disp(a), "pred": disp(b), "count": n} for (a, b), n in confusions.most_common(10)],
           "stress_test_12_objects_correct": int(round(s_obj["recall"] * 12))}
json.dump(summary, open(OUTPUT_ROOT / "report_summary.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print("\nAll outputs saved to:", OUTPUT_ROOT)